### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
# cd C:\Users\ongk2\Documents\UCLA\ECE247_Python_Winter2026\FinalProject\emg2qwerty

In [1]:
cd '/content/drive/MyDrive/emg2qwerty'

/content/drive/MyDrive/emg2qwerty


### Step 3: Install required packages

After installing them, Colab will require you to restart the session.

In [3]:
pip install -r requirements.txt

     | 553.6 kB 3.1 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 995.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 7.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could

### Step 4: Start your experiments!

- Remember to download and copy the dataset to this directory: `Your_Dir/emg2qwerty/data`.
- You may now start your experiments with any scripts! Below are examples of single-user training and testing (greedy decoding).
- **There are two ways to track the logs:**
  - 1. Keep `--multirun`, and the logs will not be printed here, but they will be saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/submitit_logs/`.
  - 2. Comment out `--multirun` and the logs will be printed in this notebook, but they will not be saved.

#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [2]:
import sys
import torch
import torchvision
import torchaudio
import re

print("python version:", sys.version)
print("python version info:", sys.version_info)
print("torch version:", torch.__version__)
print("cuda version (torch):", torch.version.cuda)
print("torchvision version:", torchvision.__version__)
print("torchaudio version:", torchaudio.__version__)
print("cuda available:", torch.cuda.is_available())
import os
os.environ["HYDRA_FULL_ERROR"] = "1"

python version: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
python version info: sys.version_info(major=3, minor=12, micro=12, releaselevel='final', serial=0)
torch version: 2.3.0+cu121
cuda version (torch): 12.1
torchvision version: 0.18.0+cu121
torchaudio version: 2.3.0+cu121
cuda available: True


### Other Model (set max_epochs as necessary):

- tds_lstm_ctc.yaml
- tds_lstm_hidSz_48_num_5_drop_0p0_ctc.yaml
- tds_lstm_hidSz_96_num_1_drop_0p1_ctc.yaml
- tds_lstm_hidSz_96_num_3_drop_0p0_ctc.yaml
- tds_lstm_hidSz_96_num_3_drop_0p1_ctc.yaml
- tds_lstm_hidSz_96_num_3_drop_0p2_ctc.yaml
- tds_lstm_hidSz_96_num_4_drop_0p2_ctc.yaml
- tds_lstm_hidSz_96_num_4_drop_0p4_ctc.yaml
- tds_lstm_hidSz_96_num_8_drop_0p1_ctc.yaml
- tds_lstm_hidSz_192_num_4_drop_0p05_ctc.yaml
- tds_lstm_hidSz_384_num_4_drop_0p1_ctc.yaml

In [3]:
# Single-user training
!python -m emg2qwerty.train \
  user="single_user" model="tds_lstm_ctc" \
  module.hidden_size=96 module.num_layers=4 module.dropout=0.4 \
  trainer.max_epochs=100 \
  trainer.accelerator=gpu trainer.devices=1

[2026-03-13 17:39:22,277][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user="single_user" model="tds_lstm_ctc" \
  'checkpoint="/content/drive/MyDrive/emg2qwerty/logs/epoch=95-step=11520.ckpt"' \
  train=False trainer.accelerator=gpu \
  decoder=ctc_greedy

[2026-03-13 11:48:00,123][__main__][INFO] - 
Config:
user: single_user
dataset:
  train:
  - user: 89335547
    session: 2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627003020-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-21-1626916256-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-07-22-1627004019-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f
  - user: 89335547
    session: 2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f

In [4]:
from google.colab import runtime
runtime.unassign()